# Análise de Sentimentos: validação do classificador em amostra anotada

**Modelo avaliado:** `cardiffnlp/twitter-xlm-roberta-base-sentiment` (XLM-RoBERTa), como usado na Gold `governor_sentiment` da Coleta do TCC.
**Seção do TCC:** 5.4 (Análise de Sentimentos). **Arquivos:** pasta [`validacao_sentimento/`](validacao_sentimento/).

## 1. Pergunta

O classificador rotulou 74.865 comentários como `positive`, `neutral` ou `negative` sem que nenhum deles tivesse sido conferido.
A *North Star Metric*, a dimensão Qualidade do escore composto e o achado de que a negatividade cresce nas publicações de maior
alcance dependem desses rótulos. Este notebook responde, com uma amostra anotada:

1. Quando o modelo diz `positive`, `neutral` ou `negative`, com que frequência acerta (**precisão por classe**)?
2. Qual é a **acurácia**, a **revocação** e o **F1** por classe na população de comentários, e o **F1-macro**?
3. O erro do modelo **desloca a proporção de positivos** (a aprovação que entra na NSM)? Em que direção?
4. O modelo se comporta de outro modo nos comentários de **Reels virais**, onde está o achado central?

## 2. Desenho

| Decisão | Escolha | Motivo |
|---|---|---|
| População | 74.865 comentários classificados (legendas e os 769 comentários sem rótulo ficam fora) | É a base das proporções de sentimento do TCC |
| Amostra principal | 300 comentários, 100 por **classe prevista**, sorteio aleatório simples dentro da classe | A classe `negative` é 7,8% da população: um sorteio simples de 300 traria cerca de 23 negativos, pouco para medir a precisão justamente da classe que sustenta o achado |
| Suplemento viral | 60 comentários, 20 por classe prevista, só de Reels do grupo viral (27 vídeos) | Um sorteio da população traria um ou dois comentários de virais |
| Anotação às cegas | O sorteio grava o texto e o rótulo do modelo em **arquivos separados**; a anotação só viu o texto | Evita ancoragem no rótulo do modelo |
| Estimativa | Precisão lida direto da amostra; acurácia, revocação, F1 e prevalências **pós-estratificadas** pelos pesos das classes previstas na população | A amostra é balanceada por classe prevista e a população não é; sem os pesos, a acurácia e a revocação ficariam distorcidas |
| Incerteza | Intervalo de Wilson para as precisões; *bootstrap* estratificado (2.000 reamostragens) para as demais | n = 100 por estrato |

**Quem anotou.** Os 360 comentários foram anotados por um modelo de linguagem (Claude, da Anthropic), às cegas e segundo o
critério escrito da Seção 4, e a anotação é **conferida pelo autor** na planilha `validacao_sentimento/planilha_conferencia.xlsx`.
O notebook usa a coluna `rotulo_revisado` sempre que ela estiver preenchida e informa quantos rótulos foram alterados na conferência.
Não há segundo anotador humano independente nem medida de concordância entre anotadores: isso é uma limitação e deve ser dito assim no texto.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, cohen_kappa_score
from statsmodels.stats.proportion import proportion_confint

RAIZ = Path.cwd().resolve().parents[1]
PASTA = Path.cwd().resolve() / "validacao_sentimento"
sys.path.insert(0, str(RAIZ))
sys.path.insert(0, str(PASTA))

import amostragem as am
from src.repositories.delta_repository import DeltaRepository

pd.set_option("display.max_colwidth", 160)
CLASSES = am.CLASSES
repo = DeltaRepository(RAIZ / "data" / "gold", RAIZ / "data" / "silver", RAIZ / "data" / "bronze")

## 3. População e sorteio

A população é lida da Gold pelo `DeltaRepository`. A tabela abaixo dá o tamanho de cada classe prevista, que é o **peso** usado
na pós-estratificação, e quantos desses comentários pertencem a Reels do grupo viral.

In [2]:
pop = am.carregar_populacao(repo)

resumo_pop = pd.DataFrame({
    "comentários": pop["sentiment_label"].value_counts().reindex(CLASSES),
    "em Reels virais": pop[pop["viral"]]["sentiment_label"].value_counts().reindex(CLASSES),
})
resumo_pop["peso na população"] = resumo_pop["comentários"] / resumo_pop["comentários"].sum()
resumo_pop["peso entre os virais"] = resumo_pop["em Reels virais"] / resumo_pop["em Reels virais"].sum()
resumo_pop.loc["total"] = resumo_pop.sum()
resumo_pop.style.format({"comentários": "{:,.0f}", "em Reels virais": "{:,.0f}",
                         "peso na população": "{:.1%}", "peso entre os virais": "{:.1%}"})

,comentários,em Reels virais,peso na população,peso entre os virais
sentiment_label,,,,
positive,"45,795",113,61.2%,51.1%
neutral,"23,199",61,31.0%,27.6%
negative,"5,871",47,7.8%,21.3%
total,"74,865",221,100.0%,100.0%


O sorteio está em [`validacao_sentimento/amostragem.py`](validacao_sentimento/amostragem.py) e tem semente fixa. A célula abaixo
**refaz o sorteio** e confere que ele reproduz exatamente os arquivos gravados antes da anotação, isto é, que a amostra anotada é a
que o código sorteia e não uma escolha posterior.

In [3]:
amostra = am.sortear(pop)
gabarito = pd.read_csv(am.ARQ_GABARITO)
textos = pd.read_csv(am.ARQ_AMOSTRA)

assert list(amostra["id_comment"].astype(str)) == list(gabarito["id_comment"].astype(str)), "o sorteio não reproduz o gabarito"
assert list(amostra["text"]) == list(textos["text"]), "o sorteio não reproduz a amostra anotada"
print(f"Sorteio reproduzido: {len(amostra)} comentários, semente {am.SEED}.")

pd.crosstab(gabarito["estrato"], gabarito["sentiment_label"], margins=True, margins_name="total")[CLASSES + ["total"]]

Sorteio reproduzido: 360 comentários, semente 20261119.


sentiment_label,positive,neutral,negative,total
estrato,,,,
principal,100,100,100,300
viral,20,20,20,60
total,120,120,120,360


In [4]:
print("Publicações distintas na amostra principal:", gabarito.query("estrato == 'principal'")["id_reel"].nunique(), "de", pop["id_reel"].nunique(), "na população")
print("Reels virais representados no suplemento:", gabarito.query("estrato == 'viral'")["id_reel"].nunique(), "de", pop.loc[pop["viral"], "id_reel"].nunique())
print("Origem dos comentários da amostra principal:")
print(gabarito.query("estrato == 'principal'")["origem_comentario"].value_counts().to_string())
print("Textos distintos na amostra:", textos["text"].nunique(), "de", len(textos))

Publicações distintas na amostra principal: 296 de 8466 na população
Reels virais representados no suplemento: 25 de 27
Origem dos comentários da amostra principal:
origem_comentario
reel    213
post     87
Textos distintos na amostra: 320 de 360


## 4. Critério de anotação

O rótulo humano responde a uma pergunta só: **que sentimento o texto do comentário expressa?** É a mesma pergunta que o
classificador responde. O anotador não sabe em qual perfil o comentário foi feito, e por isso o rótulo depende apenas do texto.

| Rótulo | Quando usar |
|---|---|
| `positive` | Elogio, apoio, comemoração, bênção, declaração de voto ou de adesão ("estou com 15", "bora"), emojis de aplauso, coração, foguete, confirmação (✅) |
| `negative` | Crítica, insulto, rejeição ("fora X"), denúncia, lamento, deboche ou ironia contra alguém, **cobrança** por omissão ou por serviço que não funciona, emojis de reprovação, tristeza ou nojo |
| `neutral` | Texto sem carga avaliativa: fato, pergunta sem indignação, **pedido** educado de ajuda, saudação, nome ou @ solto, propaganda, e os casos em que a polaridade **não pode ser decidida só pelo texto** |

Casos difíceis, decididos antes de anotar e marcados na coluna `tags` para as análises de sensibilidade da Seção 8:

| Tag | Caso | Decisão |
|---|---|---|
| `numero` | Número de candidato sem nenhum marcador afetivo ("22", "13 13", "Zema30") | `neutral`: sem saber de quem é o perfil, o mesmo número pode ser apoio ou provocação. Com marcador afetivo ("5️⃣5️⃣💙", "13 ❤️", "44 ✅") é `positive` e não leva a tag |
| `cobranca` | Cobrança ou protesto ("E os Guarani Kaiowá?", "cadê nossos salários") | `negative` |
| `pedido` | Pedido educado, sem acusação (ajuda para exame, convocação de concurso, PIX) | `neutral` |
| `terceiro` | O alvo do sentimento é um terceiro (Lula, STF, adversário, prefeitura), não o dono do perfil | Rotula-se o sentimento do texto. A tag permite medir quanto isso pesa quando o rótulo é lido como aprovação do governador |
| `ironia` | Ironia, sarcasmo, deboche | Rotula-se o sentido pretendido, não o literal |
| `misto` | Dois sentimentos no mesmo texto | O dominante; se nenhum domina, `neutral` |
| `ambiguo` | O anotador não tem segurança (emoji isolado de riso ou choro, texto truncado, gíria) | Rotula-se a leitura mais provável; emoji de riso isolado é `neutral` |

## 5. Rótulos anotados e conferência

`rotulos_anotador.csv` guarda a anotação original. A planilha de conferência é criada uma única vez (para não sobrescrever a
revisão) e **não contém o rótulo do modelo**, de modo que a conferência também é feita às cegas. Vale o `rotulo_revisado` quando
preenchido; caso contrário, o rótulo do anotador.

In [5]:
rotulos = pd.read_csv(PASTA / "rotulos_anotador.csv", keep_default_na=False)
assert set(rotulos["rotulo"]) <= set(CLASSES) and len(rotulos) == len(textos)

ARQ_PLANILHA = PASTA / "planilha_conferencia.xlsx"
if not ARQ_PLANILHA.exists():
    planilha = textos.merge(rotulos, on="id_amostra").rename(columns={"rotulo": "rotulo_anotador"})
    planilha["rotulo_revisado"] = ""
    planilha["comentario_revisor"] = ""
    with pd.ExcelWriter(ARQ_PLANILHA, engine="openpyxl") as xl:
        planilha.to_excel(xl, index=False, sheet_name="conferencia")
        ws = xl.sheets["conferencia"]
        for coluna, largura in zip("ABCDEF", (11, 100, 16, 24, 16, 40)):
            ws.column_dimensions[coluna].width = largura
        ws.freeze_panes = "A2"
    print("Planilha de conferência criada:", ARQ_PLANILHA.name)

planilha = pd.read_excel(ARQ_PLANILHA, dtype=str, keep_default_na=False)
revisado = planilha["rotulo_revisado"].str.strip().str.lower()
invalidos = planilha.loc[(revisado != "") & ~revisado.isin(CLASSES), ["id_amostra", "rotulo_revisado"]]
assert invalidos.empty, f"rótulos revisados fora de {CLASSES}:\n{invalidos}"

planilha["real"] = np.where(revisado != "", revisado, planilha["rotulo_anotador"])
n_conferidos = int((revisado != "").sum())
n_alterados = int(((revisado != "") & (revisado != planilha["rotulo_anotador"])).sum())
print(f"Linhas com rotulo_revisado preenchido: {n_conferidos} de {len(planilha)}; rótulos alterados na conferência: {n_alterados}")

dados = (
    planilha[["id_amostra", "text", "real", "tags"]]
    .merge(gabarito.rename(columns={"sentiment_label": "previsto"}), on="id_amostra")
)
principal = dados[dados["estrato"] == "principal"].copy()
viral = dados[dados["estrato"] == "viral"].copy()
print(f"Amostra principal: {len(principal)}; suplemento viral: {len(viral)}")

Linhas com rotulo_revisado preenchido: 0 de 360; rótulos alterados na conferência: 0
Amostra principal: 300; suplemento viral: 60


## 6. Resultado na amostra principal

### 6.1 Matriz de confusão

Linhas são a classe **prevista** pelo modelo (100 comentários cada); colunas são o rótulo anotado. Como cada linha tem 100
comentários, a contagem é também o percentual, e a diagonal é a **precisão** da classe.

In [6]:
def matriz(df):
    return pd.crosstab(df["previsto"], df["real"]).reindex(index=CLASSES, columns=CLASSES, fill_value=0)

M = matriz(principal)
M.rename_axis(index="previsto (modelo)", columns="anotado").style.background_gradient(cmap="Blues", axis=None)

anotado,positive,neutral,negative
previsto (modelo),,,
positive,94,5,1
neutral,73,23,4
negative,10,14,76


### 6.2 Precisão por classe

Precisão = entre os comentários que o modelo colocou na classe, a fração que o anotador também colocou. Intervalo de Wilson a 95%.

In [7]:
def tabela_precisao(df):
    linhas = []
    for c in CLASSES:
        grupo = df[df["previsto"] == c]
        acertos, n = int((grupo["real"] == c).sum()), len(grupo)
        inf, sup = proportion_confint(acertos, n, method="wilson")
        linhas.append({"classe prevista": c, "n": n, "acertos": acertos, "precisão": acertos / n, "IC 95% inf": inf, "IC 95% sup": sup})
    return pd.DataFrame(linhas).set_index("classe prevista")

precisao = tabela_precisao(principal)
precisao.style.format({"precisão": "{:.1%}", "IC 95% inf": "{:.1%}", "IC 95% sup": "{:.1%}"})

,n,acertos,precisão,IC 95% inf,IC 95% sup
classe prevista,,,,,
positive,100,94,94.0%,87.5%,97.2%
neutral,100,23,23.0%,15.8%,32.2%
negative,100,76,76.0%,66.8%,83.3%


### 6.3 Métricas na população (pós-estratificadas)

A amostra tem um terço de cada classe prevista; a população tem 61% de `positive`, 31% de `neutral` e 8% de `negative`.
Para falar da população, cada linha da matriz é convertida em proporções e multiplicada pelo peso da sua classe prevista:

$$\hat{J}_{c,k} = \frac{N_c}{N}\cdot\frac{M_{c,k}}{n_c}$$

$\hat{J}$ é a matriz de confusão estimada da população (soma 1). Dela saem a acurácia ($\sum_k \hat{J}_{k,k}$), a revocação de
cada classe ($\hat{J}_{k,k} / \sum_c \hat{J}_{c,k}$), o F1 e a **prevalência real estimada** de cada classe ($\sum_c \hat{J}_{c,k}$).

In [8]:
def pesos(populacao):
    return (populacao["sentiment_label"].value_counts(normalize=True)).reindex(CLASSES).to_numpy()

def metricas(M, w):
    M = np.asarray(M, dtype=float)
    J = (M / M.sum(axis=1, keepdims=True)) * w[:, None]
    prec = np.diag(J) / J.sum(axis=1)
    rev = np.diag(J) / J.sum(axis=0)
    f1 = 2 * prec * rev / (prec + rev)
    return {"J": J, "acuracia": np.trace(J), "precisao": prec, "revocacao": rev, "f1": f1,
            "f1_macro": f1.mean(), "prevalencia_real": J.sum(axis=0)}

def bootstrap(df, w, n_boot=2000, seed=am.SEED):
    rng = np.random.default_rng(seed)
    grupos = {c: df.loc[df["previsto"] == c, "real"].to_numpy() for c in CLASSES}
    saida = []
    for _ in range(n_boot):
        Mb = np.zeros((3, 3))
        for i, c in enumerate(CLASSES):
            reamostra = rng.choice(grupos[c], size=len(grupos[c]), replace=True)
            Mb[i] = [(reamostra == k).sum() for k in CLASSES]
        with np.errstate(invalid="ignore", divide="ignore"):
            m = metricas(Mb, w)
        saida.append(np.concatenate([[m["acuracia"], m["f1_macro"]], m["revocacao"], m["f1"], m["prevalencia_real"]]))
    return np.nanpercentile(np.array(saida), [2.5, 97.5], axis=0)

W = pesos(pop)
R = metricas(M, W)
IC = bootstrap(principal, W)

def ic(i):
    return f"[{IC[0, i]:.1%}; {IC[1, i]:.1%}]"

por_classe = pd.DataFrame({
    "peso (previsto)": W,
    "precisão": R["precisao"],
    "revocação": R["revocacao"],
    "IC 95% revocação": [ic(2 + i) for i in range(3)],
    "F1": R["f1"],
    "IC 95% F1": [ic(5 + i) for i in range(3)],
}, index=CLASSES)
display(por_classe.style.format({"peso (previsto)": "{:.1%}", "precisão": "{:.1%}", "revocação": "{:.1%}", "F1": "{:.1%}"}))
print(f"Acurácia na população: {R['acuracia']:.1%}  IC 95% {ic(0)}")
print(f"F1-macro na população: {R['f1_macro']:.1%}  IC 95% {ic(1)}")

,peso (previsto),precisão,revocação,IC 95% revocação,F1,IC 95% F1
positive,61.2%,94.0%,71.1%,[68.5%; 73.7%],80.9%,[77.9%; 83.6%]
neutral,31.0%,23.0%,63.2%,[46.8%; 81.3%],33.7%,[23.1%; 44.4%]
negative,7.8%,76.0%,76.3%,[61.6%; 94.9%],76.2%,[66.3%; 85.2%]


Acurácia na população: 70.6%  IC 95% [66.5%; 74.5%]
F1-macro na população: 63.6%  IC 95% [58.2%; 68.9%]


In [9]:
J = pd.DataFrame(R["J"], index=CLASSES, columns=CLASSES).rename_axis(index="previsto (modelo)", columns="anotado")
J.loc["total anotado"] = J.sum()
J["total previsto"] = J.sum(axis=1)
J.style.format("{:.1%}").set_caption("Matriz de confusão estimada da população (cada célula é % dos 74.865 comentários)")

anotado,positive,neutral,negative,total previsto
previsto (modelo),,,,
positive,57.5%,3.1%,0.6%,61.2%
neutral,22.6%,7.1%,1.2%,31.0%
negative,0.8%,1.1%,6.0%,7.8%
total anotado,80.9%,11.3%,7.8%,100.0%


### 6.4 O erro desloca a aprovação?

A NSM e a dimensão Qualidade usam a **proporção de positivos**. Se o modelo erra mais para um lado, a aprovação medida fica
enviesada. A tabela compara a proporção que o modelo atribui a cada classe com a prevalência real estimada pela anotação.

In [10]:
desloc = pd.DataFrame({
    "segundo o modelo": W,
    "estimada pela anotação": R["prevalencia_real"],
    "IC 95%": [ic(8 + i) for i in range(3)],
}, index=CLASSES)
desloc["diferença (p.p.)"] = (desloc["segundo o modelo"] - desloc["estimada pela anotação"]) * 100
desloc.style.format({"segundo o modelo": "{:.1%}", "estimada pela anotação": "{:.1%}", "diferença (p.p.)": "{:+.1f}"})

,segundo o modelo,estimada pela anotação,IC 95%,diferença (p.p.)
positive,61.2%,80.9%,[76.7%; 84.8%],-19.7
neutral,31.0%,11.3%,[7.7%; 15.2%],+19.7
negative,7.8%,7.8%,[6.2%; 9.7%],+0.0


### 6.5 Métricas brutas da amostra (para registro)

O relatório abaixo trata os 300 comentários como se fossem a população. **Não é a estimativa do desempenho na população**, porque a
amostra tem um terço de cada classe prevista; fica aqui para transparência e para quem quiser comparar com a Seção 6.3. O kappa de
Cohen mede a concordância entre modelo e anotação além do acaso, também na amostra balanceada.

In [11]:
print(classification_report(principal["real"], principal["previsto"], labels=CLASSES, digits=3, zero_division=0))
print(f"Kappa de Cohen (modelo x anotação, amostra principal): {cohen_kappa_score(principal['real'], principal['previsto']):.3f}")

              precision    recall  f1-score   support

    positive      0.940     0.531     0.679       177
     neutral      0.230     0.548     0.324        42
    negative      0.760     0.938     0.840        81

    accuracy                          0.643       300
   macro avg      0.643     0.672     0.614       300
weighted avg      0.792     0.643     0.673       300

Kappa de Cohen (modelo x anotação, amostra principal): 0.465


## 7. Comentários de Reels virais

O achado central do TCC compara a negatividade dos comentários de Reels virais com a dos demais. A pergunta aqui é se a precisão
do modelo, em especial na classe `negative`, se mantém nesses comentários. O suplemento tem 20 comentários por classe prevista:
os intervalos são largos e a leitura é de **ordem de grandeza**, não de terceira casa.

In [12]:
Mv = matriz(viral)
display(Mv.rename_axis(index="previsto (modelo)", columns="anotado").style.background_gradient(cmap="Blues", axis=None))
display(tabela_precisao(viral).style.format({"precisão": "{:.1%}", "IC 95% inf": "{:.1%}", "IC 95% sup": "{:.1%}"}))

anotado,positive,neutral,negative
previsto (modelo),,,
positive,18,0,2
neutral,5,10,5
negative,0,2,18


,n,acertos,precisão,IC 95% inf,IC 95% sup
classe prevista,,,,,
positive,20,18,90.0%,69.9%,97.2%
neutral,20,10,50.0%,29.9%,70.1%
negative,20,18,90.0%,69.9%,97.2%


In [13]:
Wv = pesos(pop[pop["viral"]])
Rv = metricas(Mv, Wv)
ICv = bootstrap(viral, Wv)

comparacao = pd.DataFrame({
    "todos: modelo": W,
    "todos: anotação": R["prevalencia_real"],
    "virais: modelo": Wv,
    "virais: anotação": Rv["prevalencia_real"],
    "virais: IC 95%": [f"[{ICv[0, 8 + i]:.1%}; {ICv[1, 8 + i]:.1%}]" for i in range(3)],
}, index=CLASSES)
display(comparacao.style.format({c: "{:.1%}" for c in comparacao.columns if "IC" not in c}))
print(f"Razão de negativos virais/todos segundo o modelo:   {Wv[2] / W[2]:.2f}")
print(f"Razão de negativos virais/todos segundo a anotação: {Rv['prevalencia_real'][2] / R['prevalencia_real'][2]:.2f}")

,todos: modelo,todos: anotação,virais: modelo,virais: anotação,virais: IC 95%
positive,61.2%,80.9%,51.1%,52.9%,[43.7%; 61.0%]
neutral,31.0%,11.3%,27.6%,15.9%,[9.3%; 22.5%]
negative,7.8%,7.8%,21.3%,31.2%,[23.1%; 41.1%]


Razão de negativos virais/todos segundo o modelo:   2.71
Razão de negativos virais/todos segundo a anotação: 3.99


## 8. Onde o modelo erra

### 8.1 Discordância por tipo de caso

Taxa de discordância entre modelo e anotação por tag do critério (amostra principal e suplemento juntos, 360 comentários; um
comentário pode ter mais de uma tag).

In [14]:
dados["erro"] = dados["previsto"] != dados["real"]
tags = dados.assign(tag=dados["tags"].replace("", "sem tag").str.split(";")).explode("tag")
por_tag = tags.groupby("tag")["erro"].agg(n="size", discordâncias="sum", taxa="mean").sort_values("n", ascending=False)
por_tag.style.format({"taxa": "{:.0%}"})

,n,discordâncias,taxa
tag,,,
sem tag,242,87,36%
ambiguo,43,20,47%
cobranca,29,5,17%
terceiro,19,4,21%
numero,18,2,11%
ironia,13,4,31%
misto,8,5,62%
pedido,5,4,80%


### 8.2 Sensibilidade ao critério

Duas decisões do critério são discutíveis e mudam o resultado. A tabela recalcula as métricas da população em três cenários:

- **critério adotado**: o da Seção 4;
- **número = apoio**: os comentários com a tag `numero` passam a `positive` (leitura de que digitar o número do candidato é sempre apoio);
- **sem ambíguos**: os comentários com a tag `ambiguo` saem da amostra.

In [15]:
def cenario(df, nome):
    r = metricas(matriz(df), W)
    return {"cenário": nome, "n": len(df), "acurácia": r["acuracia"], "F1-macro": r["f1_macro"],
            "precisão negative": r["precisao"][2], "revocação negative": r["revocacao"][2],
            "precisão positive": r["precisao"][0], "revocação positive": r["revocacao"][0],
            "positivos (anotação)": r["prevalencia_real"][0]}

tem = lambda df, tag: df["tags"].str.split(";").apply(lambda t: tag in t)
numero_apoio = principal.assign(real=np.where(tem(principal, "numero"), "positive", principal["real"]))
sem_ambiguos = principal[~tem(principal, "ambiguo")]

sens = pd.DataFrame([
    cenario(principal, "critério adotado"),
    cenario(numero_apoio, "número = apoio"),
    cenario(sem_ambiguos, "sem ambíguos"),
]).set_index("cenário")
sens.style.format({c: "{:.1%}" for c in sens.columns if c != "n"})

,n,acurácia,F1-macro,precisão negative,revocação negative,precisão positive,revocação positive,positivos (anotação)
cenário,,,,,,,,
critério adotado,300,70.6%,63.6%,76.0%,76.3%,94.0%,71.1%,80.9%
número = apoio,300,65.9%,55.9%,76.0%,76.3%,94.0%,67.1%,85.7%
sem ambíguos,269,70.3%,63.1%,84.1%,76.5%,94.9%,70.3%,82.6%


### 8.3 Sentimento do texto × aprovação do governador

O classificador mede o sentimento do texto, e o TCC lê `positive` como aprovação. A tag `terceiro` marca os comentários em que o
alvo do sentimento não é o dono do perfil. A tabela mostra quanto de cada classe **prevista** tem alvo em terceiro: é a parcela em
que, mesmo com o modelo certo, "negativo" não significa "contra o governador".

In [16]:
principal["alvo_terceiro"] = tem(principal, "terceiro")
terceiro = principal.groupby("previsto")["alvo_terceiro"].agg(n="size", com_alvo_em_terceiro="sum", fração="mean").reindex(CLASSES)
display(terceiro.style.format({"fração": "{:.0%}"}))
print(f"Na população (ponderado): {float((terceiro['fração'].to_numpy() * W).sum()):.1%} dos comentários têm alvo em terceiro.")

,n,com_alvo_em_terceiro,fração
previsto,,,
positive,100,0,0%
neutral,100,0,0%
negative,100,11,11%


Na população (ponderado): 0.9% dos comentários têm alvo em terceiro.


### 8.4 Todas as discordâncias

Lista completa dos comentários em que modelo e anotação divergem, para leitura direta.

In [17]:
with pd.option_context("display.max_rows", 400, "display.max_colwidth", 220):
    display(dados.loc[dados["erro"], ["id_amostra", "estrato", "previsto", "real", "tags", "text"]]
            .sort_values(["previsto", "real", "id_amostra"]).reset_index(drop=True))

[saída omitida na versão versionada: mostra textos reais de comentários]


## 9. Regra do Código de campanha (ADR 0048)

A Seção 6 mostra que o erro do classificador se concentra em uma célula: apoio lido como `neutral`. A Seção 8.4 mostra a forma
desse apoio: número de candidato e emojis de apoio, o **Código de campanha**. A primeira correção foi um pós-processamento
determinístico, em [`src/modeling/sentiment_rules.py`](../../src/modeling/sentiment_rules.py):

> Um comentário rotulado `neutral` pelo classificador passa a `positive` quando o texto tem ao menos um emoji de apoio, não tem
> emoji de reprovação e, ou não tem letras (só números e emojis), ou tem número e no máximo seis palavras.

A regra nunca altera `positive` nem `negative`. Número de candidato sem emoji de apoio ("22", "13 13") continua `neutral`, como no
critério de anotação.

> **Depois deste notebook**, a regra foi comparada com o ajuste fino do classificador em um teste reservado
> ([`sentimento_02_comparacao_de_metodos.ipynb`](sentimento_02_comparacao_de_metodos.ipynb)) e perdeu. A Gold usa hoje o
> classificador ajustado (ADR 0049); esta seção fica como registro da regra, que é um dos três métodos comparados.

### 9.1 Alcance da regra

In [18]:
from src.modeling.sentiment_rules import EMOJIS_DE_APOIO, EMOJIS_DE_REPROVACAO, MAX_PALAVRAS, is_codigo_de_campanha

print("Emojis de apoio:", " ".join(sorted(EMOJIS_DE_APOIO)))
print("Emojis de reprovação:", " ".join(sorted(EMOJIS_DE_REPROVACAO)))
print("Máximo de palavras quando há letras:", MAX_PALAVRAS)

pop["codigo_campanha"] = pop["text"].map(is_codigo_de_campanha)
promovido = (pop["sentiment_label"] == "neutral") & pop["codigo_campanha"]

alcance = pd.DataFrame({
    "rótulo do modelo": pop["sentiment_label"].value_counts().reindex(CLASSES),
    "em Código de campanha": pop.groupby("sentiment_label")["codigo_campanha"].sum().reindex(CLASSES),
    "rótulo após a regra": pop["sentiment_label_regra"].value_counts().reindex(CLASSES),
})
alcance["% após a regra"] = alcance["rótulo após a regra"] / alcance["rótulo após a regra"].sum()
print(f"Comentários promovidos de neutral a positive: {int(promovido.sum()):,} ({promovido.sum() / (pop['sentiment_label'] == 'neutral').sum():.1%} dos neutros)")
alcance.style.format({"rótulo do modelo": "{:,.0f}", "em Código de campanha": "{:,.0f}", "rótulo após a regra": "{:,.0f}", "% após a regra": "{:.1%}"})

Emojis de apoio: ♥ ✅ ✌ ❤ ⭐ 🎉 👍 👏 💙 💚 💛 💜 💪 💯 😍 😘 🙌 🙏 🚀 🤍 🤝 🥰 🧡
Emojis de reprovação: 👎 💔 💩 😔 😞 😠 😡 😢 😤 😭 🤡 🤢 🤬 🤮
Máximo de palavras quando há letras: 6


Comentários promovidos de neutral a positive: 8,209 (35.4% dos neutros)


,rótulo do modelo,em Código de campanha,rótulo após a regra,% após a regra
positive,"45,795","30,546","54,004",72.1%
neutral,"23,199","8,209","14,990",20.0%
negative,"5,871",9,"5,871",7.8%


### 9.2 Validação da regra em uma segunda amostra

A regra foi desenhada olhando a primeira amostra, então o acerto dela ali (primeira tabela abaixo) é otimista. Para medir a
precisão sem esse vício, foram sorteados **100 comentários atingidos pela regra**, fora da primeira amostra, e anotados pelo mesmo
critério. A planilha `planilha_conferencia_regra.xlsx` segue o mesmo esquema de conferência da Seção 5.

In [19]:
principal["codigo_campanha"] = principal["text"].map(is_codigo_de_campanha)
print("Primeira amostra (principal), comentários previstos como neutral:")
display(pd.crosstab(principal.loc[principal["previsto"] == "neutral", "codigo_campanha"],
                    principal.loc[principal["previsto"] == "neutral", "real"]).reindex(columns=CLASSES, fill_value=0)
        .rename_axis(index="em Código de campanha", columns="anotado"))

candidatos = pop[promovido & ~pop["id_comment"].astype(str).isin(set(gabarito["id_comment"].astype(str)))]
segunda = candidatos.sample(100, random_state=am.SEED + 100).reset_index(drop=True)
amostra_regra = pd.read_csv(PASTA / "amostra_regra.csv")
assert list(segunda["id_comment"].astype(str)) == list(amostra_regra["id_comment"].astype(str)), "o sorteio não reproduz a segunda amostra"

rot_regra = pd.read_csv(PASTA / "rotulos_regra.csv", keep_default_na=False)
ARQ_PLANILHA_REGRA = PASTA / "planilha_conferencia_regra.xlsx"
if not ARQ_PLANILHA_REGRA.exists():
    p2 = amostra_regra[["id_amostra", "text"]].merge(rot_regra, on="id_amostra").rename(columns={"rotulo": "rotulo_anotador"})
    p2["rotulo_revisado"] = ""
    p2["comentario_revisor"] = ""
    with pd.ExcelWriter(ARQ_PLANILHA_REGRA, engine="openpyxl") as xl:
        p2.to_excel(xl, index=False, sheet_name="conferencia")
        for coluna, largura in zip("ABCDEF", (11, 70, 16, 16, 16, 40)):
            xl.sheets["conferencia"].column_dimensions[coluna].width = largura
p2 = pd.read_excel(ARQ_PLANILHA_REGRA, dtype=str, keep_default_na=False)
rev2 = p2["rotulo_revisado"].str.strip().str.lower()
assert rev2[rev2 != ""].isin(CLASSES).all()
p2["real"] = np.where(rev2 != "", rev2, p2["rotulo_anotador"])

acertos_regra, n_regra = int((p2["real"] == "positive").sum()), len(p2)
inf_regra, sup_regra = proportion_confint(acertos_regra, n_regra, method="wilson")
print(f"Segunda amostra: {acertos_regra} de {n_regra} são apoio. Precisão da regra: {acertos_regra / n_regra:.1%} (IC 95% de {inf_regra:.1%} a {sup_regra:.1%})")
print(f"Perfis representados: {amostra_regra['inputUrl'].nunique()}; textos distintos: {amostra_regra['text'].nunique()}; rótulos conferidos: {int((rev2 != '').sum())}")
display(p2.loc[p2["real"] != "positive", ["id_amostra", "text", "real", "tags"]])

[saída omitida na versão versionada: mostra textos reais de comentários]


### 9.3 Métricas depois da regra

As mesmas métricas da Seção 6.3, agora comparando a anotação com o **rótulo corrigido**. A amostra continua estratificada pelo
rótulo original do modelo, e é por esses estratos que a ponderação é feita. Como a regra nasceu desta amostra, estes números são um
teto; a medida independente é a precisão da Seção 9.2.

In [20]:
principal["previsto_regra"] = np.where((principal["previsto"] == "neutral") & principal["codigo_campanha"], "positive", principal["previsto"])

def metricas_J(J):
    prec = np.diag(J) / J.sum(axis=1)
    rev = np.diag(J) / J.sum(axis=0)
    f1 = 2 * prec * rev / (prec + rev)
    return {"acuracia": np.trace(J), "precisao": prec, "revocacao": rev, "f1": f1, "f1_macro": f1.mean(),
            "prevalencia_prevista": J.sum(axis=1), "prevalencia_real": J.sum(axis=0)}

def J_corrigida(df, w, coluna="previsto_regra"):
    J = np.zeros((3, 3))
    for i, c in enumerate(CLASSES):
        estrato = df[df["previsto"] == c]
        J += w[i] * pd.crosstab(estrato[coluna], estrato["real"]).reindex(index=CLASSES, columns=CLASSES, fill_value=0).to_numpy() / len(estrato)
    return J

def bootstrap_regra(df, w, n_boot=2000, seed=am.SEED):
    rng = np.random.default_rng(seed)
    estratos = {c: df[df["previsto"] == c] for c in CLASSES}
    saida = []
    for _ in range(n_boot):
        re = pd.concat([g.iloc[rng.integers(0, len(g), len(g))] for g in estratos.values()])
        m = metricas_J(J_corrigida(re, w))
        saida.append([m["acuracia"], m["f1_macro"]])
    return np.percentile(np.array(saida), [2.5, 97.5], axis=0)

RR = metricas_J(J_corrigida(principal, W))
ICR = bootstrap_regra(principal, W)

antes_depois = pd.DataFrame({
    ("precisão", "modelo"): R["precisao"], ("precisão", "com a regra"): RR["precisao"],
    ("revocação", "modelo"): R["revocacao"], ("revocação", "com a regra"): RR["revocacao"],
    ("F1", "modelo"): R["f1"], ("F1", "com a regra"): RR["f1"],
    ("proporção", "modelo"): W, ("proporção", "com a regra"): RR["prevalencia_prevista"], ("proporção", "anotação"): RR["prevalencia_real"],
}, index=CLASSES)
display(antes_depois.style.format("{:.1%}"))
print(f"Acurácia:  modelo {R['acuracia']:.1%}  ->  com a regra {RR['acuracia']:.1%}  IC 95% [{ICR[0, 0]:.1%}; {ICR[1, 0]:.1%}]")
print(f"F1-macro:  modelo {R['f1_macro']:.1%}  ->  com a regra {RR['f1_macro']:.1%}  IC 95% [{ICR[0, 1]:.1%}; {ICR[1, 1]:.1%}]")

Acurácia:  modelo 70.6%  ->  com a regra 85.2%  IC 95% [81.0%; 89.0%]
F1-macro:  modelo 63.6%  ->  com a regra 73.2%  IC 95% [66.8%; 79.4%]


### 9.4 Efeito por perfil

O viés do classificador não era igual entre os perfis: dependia de quanto do apoio vinha em Código de campanha. A tabela compara,
para os perfis com ao menos cem comentários, a aprovação e a posição antes e depois da regra. A aprovação usa o total de
comentários do perfil no denominador, como a NSM; as duas NSM são recalculadas aqui, com o rótulo de cada método e o mesmo alcance
médio (a Gold guarda hoje o rótulo do classificador ajustado, ADR 0049).

In [21]:
nsm = repo.load_nsm().set_index("inputUrl")
por_perfil = pd.DataFrame({
    "perfil": nsm["username"],
    "comentários": nsm["n_comentarios_totais"],
    "em Código de campanha (%)": pop[promovido].groupby("inputUrl").size().reindex(nsm.index).fillna(0) / nsm["n_comentarios_totais"] * 100,
    "aprovação: modelo": pop[pop["sentiment_label"] == "positive"].groupby("inputUrl").size().reindex(nsm.index).fillna(0) / nsm["n_comentarios_totais"],
    "aprovação: com a regra": pop[pop["sentiment_label_regra"] == "positive"].groupby("inputUrl").size().reindex(nsm.index).fillna(0) / nsm["n_comentarios_totais"],
})
por_perfil = por_perfil[por_perfil["comentários"] >= 100].copy()
alcance_medio = nsm.loc[por_perfil.index, "alcance_medio"]
por_perfil["diferença (p.p.)"] = (por_perfil["aprovação: com a regra"] - por_perfil["aprovação: modelo"]) * 100
posto = lambda s: s.rank(ascending=False).astype(int)
por_perfil["posição aprovação: modelo"] = posto(por_perfil["aprovação: modelo"])
por_perfil["posição aprovação: regra"] = posto(por_perfil["aprovação: com a regra"])
por_perfil["posição NSM: modelo"] = posto(por_perfil["aprovação: modelo"] * alcance_medio)
por_perfil["posição NSM: regra"] = posto(por_perfil["aprovação: com a regra"] * alcance_medio)

display(por_perfil.sort_values("diferença (p.p.)", ascending=False).set_index("perfil").style.format({
    "comentários": "{:,.0f}", "em Código de campanha (%)": "{:.1f}", "aprovação: modelo": "{:.1%}",
    "aprovação: com a regra": "{:.1%}", "diferença (p.p.)": "{:+.1f}"}))

sp = lambda a, b: por_perfil[a].corr(por_perfil[b], method="spearman")
print(f"Diferença de aprovação: de {por_perfil['diferença (p.p.)'].min():+.1f} a {por_perfil['diferença (p.p.)'].max():+.1f} p.p.")
print(f"Mudam de posição em aprovação: {(por_perfil['posição aprovação: modelo'] != por_perfil['posição aprovação: regra']).sum()} de {len(por_perfil)} (Spearman {sp('aprovação: modelo', 'aprovação: com a regra'):.2f})")
print(f"Mudam de posição na NSM:       {(por_perfil['posição NSM: modelo'] != por_perfil['posição NSM: regra']).sum()} de {len(por_perfil)} (Spearman {sp('posição NSM: modelo', 'posição NSM: regra'):.2f})")

,comentários,em Código de campanha (%),aprovação: modelo,aprovação: com a regra,diferença (p.p.),posição aprovação: modelo,posição aprovação: regra,posição NSM: modelo,posição NSM: regra
perfil,,,,,,,,,
wilsonlimaam,"2,093",35.7,40.5%,76.2%,+35.7,22,9,21,20
fabiogov55,"6,277",27.1,55.7%,82.8%,+27.1,15,1,18,15
ricardoferraco,"3,369",21.9,46.2%,68.2%,+21.9,21,14,20,19
clecioluis_,"4,845",20.6,61.4%,82.0%,+20.6,10,3,14,14
gladsoncameli,"2,597",16.1,63.3%,79.4%,+16.1,8,6,22,22
antoniodenariumrr,"2,026",13.0,63.9%,76.9%,+13.0,7,8,23,23
ratinho_junior,"4,862",11.6,57.3%,68.9%,+11.6,13,13,17,16
elmanofreitas,"5,395",10.6,62.7%,73.3%,+10.6,9,10,12,12
helderbarbalho,"3,019",10.3,70.1%,80.4%,+10.3,4,5,8,8


Diferença de aprovação: de +0.8 a +35.7 p.p.
Mudam de posição em aprovação: 20 de 24 (Spearman 0.71)
Mudam de posição na NSM:       9 de 24 (Spearman 0.99)


## 10. Registro das métricas

As métricas que entram no texto do TCC são gravadas em `validacao_sentimento/metricas_validacao.json`, para que o texto cite um
arquivo e não um número copiado da tela.

In [22]:
registro = {
    "modelo": "cardiffnlp/twitter-xlm-roberta-base-sentiment",
    "populacao": int(len(pop)),
    "amostra_principal": int(len(principal)),
    "suplemento_viral": int(len(viral)),
    "semente": am.SEED,
    "rotulos_conferidos": n_conferidos,
    "rotulos_alterados_na_conferencia": n_alterados,
    "matriz_amostra_principal": M.to_dict(orient="index"),
    "acuracia": {"valor": float(R["acuracia"]), "ic95": [float(IC[0, 0]), float(IC[1, 0])]},
    "f1_macro": {"valor": float(R["f1_macro"]), "ic95": [float(IC[0, 1]), float(IC[1, 1])]},
    "por_classe": {
        c: {
            "precisao": float(R["precisao"][i]),
            "precisao_ic95": [float(precisao.loc[c, "IC 95% inf"]), float(precisao.loc[c, "IC 95% sup"])],
            "revocacao": float(R["revocacao"][i]),
            "f1": float(R["f1"][i]),
            "proporcao_modelo": float(W[i]),
            "proporcao_estimada_anotacao": float(R["prevalencia_real"][i]),
        }
        for i, c in enumerate(CLASSES)
    },
    "regra_codigo_de_campanha": {
        "comentarios_promovidos": int(promovido.sum()),
        "precisao_segunda_amostra": {"acertos": acertos_regra, "n": n_regra, "ic95": [float(inf_regra), float(sup_regra)]},
        "acuracia_com_a_regra": {"valor": float(RR["acuracia"]), "ic95": [float(ICR[0, 0]), float(ICR[1, 0])]},
        "f1_macro_com_a_regra": {"valor": float(RR["f1_macro"]), "ic95": [float(ICR[0, 1]), float(ICR[1, 1])]},
        "proporcao_com_a_regra": {c: float(RR["prevalencia_prevista"][i]) for i, c in enumerate(CLASSES)},
    },
    "virais": {
        "precisao": {c: float(v) for c, v in tabela_precisao(viral)["precisão"].items()},
        "negativos_modelo": float(Wv[2]),
        "negativos_estimados_anotacao": float(Rv["prevalencia_real"][2]),
    },
}
(PASTA / "metricas_validacao.json").write_text(json.dumps(registro, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: registro[k] for k in ("acuracia", "f1_macro", "por_classe")}, ensure_ascii=False, indent=2))

{
  "acuracia": {
    "valor": 0.7058709677419355,
    "ic95": [
      0.6652467842115809,
      0.7446011721097975
    ]
  },
  "f1_macro": {
    "valor": 0.6360465875790798,
    "ic95": [
      0.5820380888041702,
      0.6890126704893391
    ]
  },
  "por_classe": {
    "positive": {
      "precisao": 0.9400000000000001,
      "precisao_ic95": [
        0.8752318455410406,
        0.9722138760368118
      ],
      "revocacao": 0.7107071905790473,
      "f1": 0.8094285442713263,
      "proporcao_modelo": 0.6117010619114406,
      "proporcao_estimada_anotacao": 0.8090518934081345
    },
    "neutral": {
      "precisao": 0.23,
      "precisao_ic95": [
        0.15843265880303448,
        0.3215438302287378
      ],
      "revocacao": 0.6316419373397448,
      "f1": 0.3372111762263457,
      "proporcao_modelo": 0.30987778000400723,
      "proporcao_estimada_anotacao": 0.11283590462833099
    },
    "negative": {
      "precisao": 0.76,
      "precisao_ic95": [
        0.667676636501855

## 11. Leitura

> Os números desta seção são os da anotação **antes da conferência do autor** (nenhum `rotulo_revisado` preenchido). Depois de
> conferir as duas planilhas, reexecute o notebook e atualize esta leitura com as tabelas acima.

**1. O classificador é confiável quando diz `positive` e razoável quando diz `negative`; a classe `neutral` não é confiável.**
A precisão é de 94% em `positive` (IC 95% de 87,5% a 97,2%), 76% em `negative` (66,8% a 83,3%) e 23% em `neutral` (15,8% a 32,2%).
Na população, a acurácia estimada é de 70,6% (66,5% a 74,5%) e o F1-macro, de 63,6% (58,2% a 68,9%).

**2. O erro tem direção: o classificador subestima a aprovação.** De cada 100 comentários que ele chama de `neutral`, 73 são apoio
segundo a anotação, quase sempre em Código de campanha: número de candidato com emoji de apoio ("5️⃣5️⃣💙🚀", "4️⃣0️⃣0️⃣✅"), foguetes,
"bora". O modelo foi treinado em tuítes e não lê esse código como apoio. Ele atribui 61,2% de positivos e a anotação estima 80,9%
(76,7% a 84,8%).

**3. A proporção de negativos não está enviesada.** O classificador atribui 7,8% de negativos e a anotação estima 7,8% (6,2% a 9,7%):
os falsos negativos e os falsos positivos da classe se compensam. Os erros de `negative` são de dois tipos: pedido educado ou texto
ambíguo lido como crítica, e apoio escrito com vocabulário de confronto ("pra cima", "bora", "neles").

**4. O achado sobre os virais resiste à validação.** Nos comentários de Reels virais a precisão de `negative` é de 18 em 20
(IC 95% de 69,9% a 97,2%), e a proporção de negativos estimada pela anotação é de 31,2% (23,1% a 41,1%), contra 7,8% no conjunto.
A razão entre virais e conjunto é de 2,7 segundo o classificador e de 4,0 segundo a anotação: o erro **não fabrica** a diferença e,
se alguma coisa, a atenua. O suplemento tem 20 comentários por classe; a conclusão é sobre a direção e a ordem de grandeza.

**5. A regra do Código de campanha corrige a maior parte do viés.** Ela promove 8.209 comentários de `neutral` a `positive` (35,4%
dos neutros). Na segunda amostra, sorteada só para validá-la, 99 de 100 comentários promovidos são apoio (IC 95% de 94,6% a 99,8%).
Com a regra, a acurácia estimada passa de 70,6% para 85,2% (81,0% a 89,0%), o F1-macro de 63,6% para 73,2% e a proporção de positivos
de 61,2% para 75,7% entre os comentários classificados. A regra não toca em `negative`, e por isso os itens 3 e 4 não mudam.

**6. O viés era desigual entre perfis, e a correção muda a aprovação por perfil mais do que a NSM.** A diferença de aprovação vai
de +0,8 ponto (Eduardo Leite, Eduardo Riedel) a +35,7 pontos (Wilson Lima); 20 dos 24 perfis mudam de posição em aprovação
(correlação de postos de 0,71), e Fábio Mitidieri passa do 15º para o 1º lugar. Na NSM, que multiplica a aprovação pelo alcance
médio, só 9 dos 24 mudam de posição (correlação de 0,99).

**7. O que a regra não corrige.** Resta um viés de cerca de cinco pontos na aprovação (75,7% com a regra, 80,9% pela anotação): é o
apoio em texto que o classificador lê como neutro e que não tem a forma do Código de campanha ("Bora lula", "Tem meu voto"). A
classe `neutral` continua a menos confiável (precisão de 43,4% com a regra). Esse resto motivou o ajuste fino do classificador,
comparado com a regra em teste reservado no notebook `sentimento_02_comparacao_de_metodos.ipynb`; o ajuste fino venceu e é o método
em uso na Gold (ADR 0049).

**8. O que depende do critério.** Se número de candidato sem emoji de apoio for lido como apoio, a acurácia do classificador cai
para 65,9% e a aprovação estimada sobe para 85,7%; a precisão e a revocação de `negative` não mudam. Em 11% dos comentários que o
classificador chama de negativos, o alvo é um terceiro (Lula, STF, adversário), não o dono do perfil: "negativo" não é sinônimo de
"contra o governador".

**Limitações.** Anotação feita por modelo de linguagem, com conferência do autor pendente, sem segundo anotador humano independente e sem
medida de concordância entre anotadores. O anotador não sabia de quem era o perfil, o que impede decidir a polaridade de um número
de candidato isolado. O suplemento viral é pequeno. As métricas "com a regra" da Seção 9.3 são um teto, porque a regra foi desenhada
sobre a mesma amostra; a medida independente é a precisão da Seção 9.2. A amostra vem de uma única Coleta, em período eleitoral, e o
Código de campanha pode ter outra forma, ou não existir, fora dele.